# MT3 — YOLO26 Entity Detector + Edge Benchmark (Kaggle)

Kaggle replacement for the Colab notebook. Same recipe as the frozen YOLOv11s baseline (fair swap).

### Setup (once)
1. **Add the dataset:** right panel → *Add Input* → *Upload* → upload `unsafe-net-detector-dataset.zip`
   as a private Kaggle Dataset (e.g. name it `unsafe-net-detector`). It mounts read-only at
   `/kaggle/input/<dataset-slug>/`.
2. **GPU:** right panel → *Session options* → Accelerator → **GPU T4 x2** (or P100).
3. **Run headless (no disconnect):** top-right → *Save Version* → **Save & Run All (Commit)**. It runs on
   Kaggle's servers with the browser closed. When done, open the version → **Output** tab → download
   `yolo26_best.pt` and `yolo26_benchmark.csv`.

For real Jetson numbers, run cell 7 on the Jetson.


## 1. GPU + Ultralytics


In [ ]:
!nvidia-smi -L
!pip -q install -U ultralytics
import torch, ultralytics; from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Locate dataset zip + unzip to working dir


In [ ]:
import glob, os, shutil
# Kaggle auto-extracts uploaded zips -> dataset is already unpacked under /kaggle/input (read-only).
hits = glob.glob('/kaggle/input/**/images/train', recursive=True)
assert hits, f'No images/train found. /kaggle/input = {os.listdir("/kaggle/input")}'
src = os.path.dirname(os.path.dirname(hits[0]))        # .../<root>/images/train -> <root>
print('dataset src (read-only):', src)
root = '/kaggle/working/ds'
if not os.path.exists(root):
    shutil.copytree(src, root)                          # writable copy for label caches
print('dataset root:', root, '| train', len(glob.glob(root+'/images/train/*.jpg')), '| val', len(glob.glob(root+'/images/val/*.jpg')))


## 3. data.yaml (nc=8, identical to the YOLO11 run)


In [ ]:
open('/kaggle/working/data.yaml','w').write(f'''path: {root}\ntrain: images/train\nval: images/val\nnc: 8\nnames:\n  0: Person_Authorized\n  1: Person_Unauthorized\n  2: Safe_Walkway\n  3: Restricted_Zone\n  4: Panel_Closed\n  5: Panel_Opened\n  6: Forklift_Safe_Load\n  7: Forklift_Overload\n''')
print(open('/kaggle/working/data.yaml').read())


## 4. Train YOLO26 — SAME recipe as YOLOv11s baseline (fair swap)
2-stage freeze→fine-tune, SGD lr 0.01→0.001, imgsz 800, hsv_h=0, mosaic 0.5, seed 0.
Output → `/kaggle/working/runs26` (persisted in the committed version's Output).


In [ ]:
RUNS='/kaggle/working/runs26'
AUG=dict(hsv_h=0.0,hsv_s=0.5,hsv_v=0.4,degrees=0.0,flipud=0.0,fliplr=0.5,translate=0.1,scale=0.5,mosaic=0.5,close_mosaic=10,mixup=0.0)
COMMON=dict(data='/kaggle/working/data.yaml',imgsz=800,batch=16,device=0,optimizer='SGD',momentum=0.937,weight_decay=0.0005,seed=0,patience=30,plots=True,project=RUNS,exist_ok=True,**AUG)
m=YOLO('yolo26s.pt')   # if unavailable, fall back to 'yolo11s.pt' and note the version
m.train(name='stage1_frozen',freeze=10,epochs=30,lr0=0.01,**COMMON)
m2=YOLO(RUNS+'/stage1_frozen/weights/best.pt')
m2.train(name='stage2_finetune',freeze=0,epochs=100,lr0=0.001,**COMMON)
BEST=RUNS+'/stage2_finetune/weights/best.pt'
import shutil; shutil.copy(BEST, '/kaggle/working/yolo26_best.pt')   # <- download this from Output tab
print('best:',BEST,'\nDOWNLOAD -> /kaggle/working/yolo26_best.pt')


## 5. Validate — compare to YOLO11 baseline (mAP@50 0.969 / 50-95 0.894)


In [ ]:
best=YOLO(BEST); mm=best.val(data='/kaggle/working/data.yaml',imgsz=800,device=0)
print('YOLO26 mAP50=',round(float(mm.box.map50),3),' mAP50-95=',round(float(mm.box.map),3),' (YOLO11 was 0.969 / 0.894)')
for i,c in enumerate(mm.box.ap_class_index): print(f'  {best.names[int(c)]:20s} mAP50={mm.box.ap50[i]:.3f} R={mm.box.r[i]:.3f}')


## 6. MT3 edge metrics — params/FLOPs, size, per-format latency, INT8


In [ ]:
best.info()
from ultralytics.utils.benchmarks import benchmark
df = benchmark(model=BEST, data='/kaggle/working/data.yaml', imgsz=800, half=True, int8=True)
print(df)
df.to_csv('/kaggle/working/yolo26_benchmark.csv')   # download from Output tab


## Next (local)
From the committed version's **Output** tab, download `yolo26_best.pt` → drop at
`sweep_results/entity_detector/yolo26_best.pt`. Then locally:
`build_relation_tokens.py --detector sweep_results/entity_detector/yolo26_best.pt --appearance
--appearance_backbone yolo` → re-run S1a → paired McNemar + LCO vs the YOLO11 baseline.
